# Basic SMOTE for Images Pipeline Usage

This notebook demonstrates the basic usage of the SMOTE for Images pipeline for synthetic image generation.

## Overview

The SMOTE for Images pipeline consists of:
1. **Encoder**: Converts images to embeddings (ResNet-based)
2. **SMOTE**: Generates synthetic embeddings using constrained SMOTE
3. **Decoder**: Converts embeddings back to images (Autoencoder/VAE)
4. **Quality Assessment**: Evaluates synthetic image quality

In [ ]:
# Import required libraries
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import SMOTE for Images components
from smote_image_synthesis.pipeline import SynthesisPipeline
from smote_image_synthesis.encoders.resnet_encoder import ResNetEncoder
from smote_image_synthesis.decoders.autoencoder_decoder import AutoencoderDecoder
from smote_image_synthesis.smote.constrained_smote import ConstrainedSMOTE
from smote_image_synthesis.quality.assessor import QualityAssessor
from smote_image_synthesis.data.models import PipelineConfig

## 1. Generate Sample Data

For this demo, we'll create synthetic sample data. In practice, you would load your own image dataset.

In [ ]:
# Generate sample data for demonstration
def generate_sample_data(n_samples=100, image_size=(3, 64, 64), n_classes=3):
    """
    Generate sample image data for demonstration.
    """
    # Create random images with some structure
    images = torch.randn(n_samples, *image_size)
    
    # Create imbalanced labels
    labels = np.random.choice(n_classes, n_samples, p=[0.7, 0.2, 0.1])
    
    # Add some class-specific patterns
    for i, label in enumerate(labels):
        if label == 0:
            images[i] += 0.5  # Brighter images for class 0
        elif label == 1:
            images[i] -= 0.3  # Darker images for class 1
        # Class 2 remains as random
    
    # Normalize to [0, 1]
    images = torch.clamp(images, 0, 1)
    
    return images, labels

# Generate sample dataset
images, labels = generate_sample_data(n_samples=150, image_size=(3, 64, 64))

print(f"Generated {len(images)} images with shape {images.shape[1:]}")
print(f"Class distribution: {np.bincount(labels)}")

## 2. Set Up Pipeline Components

Configure each component of the pipeline with appropriate parameters.

In [ ]:
# Configure encoder
encoder = ResNetEncoder(
    architecture='resnet18',  # Use smaller model for demo
    embedding_dim=128,        # Compact embedding space
    pretrained=True,          # Use pretrained weights
    freeze_backbone=True      # Freeze backbone for faster training
)

# Configure decoder
decoder = AutoencoderDecoder(
    embedding_dim=128,
    image_shape=(3, 64, 64),
    hidden_dims=[256, 512, 256],
    use_batch_norm=True
)

# Configure SMOTE
smote = ConstrainedSMOTE(
    k_neighbors=5,
    sampling_strategy='auto',  # Automatically balance classes
    use_clustering=True,       # Use semantic clustering
    clustering_method='kmeans'
)

# Configure quality assessor
quality_assessor = QualityAssessor(
    metrics=['mse', 'ssim', 'psnr'],  # Use faster metrics for demo
    compute_diversity=True
)

print("Pipeline components configured successfully!")

## 3. Create and Configure Pipeline

Combine all components into a unified pipeline.

In [ ]:
# Create pipeline
pipeline = SynthesisPipeline(
    encoder=encoder,
    decoder=decoder,
    smote=smote,
    quality_assessor=quality_assessor,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
)

print(f"Pipeline created on device: {pipeline.device}")

## 4. Train the Pipeline

Fit the pipeline on the training data. This includes training the decoder if needed.

In [ ]:
# Split data into train/validation
train_size = int(0.8 * len(images))
train_images = images[:train_size]
train_labels = labels[:train_size]
val_images = images[train_size:]
val_labels = labels[train_size:]

print(f"Training set: {len(train_images)} images")
print(f"Validation set: {len(val_images)} images")

# Fit the pipeline
print("\nFitting pipeline...")
pipeline.fit(
    images=train_images,
    labels=train_labels,
    train_decoder=True,  # Train the decoder
    decoder_epochs=10,   # Quick training for demo
    validation_data=(val_images, val_labels)
)

print("Pipeline fitted successfully!")

## 5. Generate Synthetic Images

Use the trained pipeline to generate synthetic images for minority classes.

In [ ]:
# Generate synthetic images
n_synthetic = 50
print(f"Generating {n_synthetic} synthetic images...")

synthetic_images, synthetic_labels = pipeline.generate_synthetic_images(
    n_samples=n_synthetic,
    target_classes=None  # Generate for all minority classes
)

print(f"Generated {len(synthetic_images)} synthetic images")
print(f"Synthetic class distribution: {np.bincount(synthetic_labels)}")

## 6. Visualize Results

Compare original and synthetic images to assess quality.

In [ ]:
# Visualize original vs synthetic images
def plot_image_comparison(real_images, synthetic_images, real_labels, synthetic_labels, n_samples=8):
    """
    Plot comparison between real and synthetic images.
    """
    fig, axes = plt.subplots(2, n_samples, figsize=(16, 4))
    
    # Plot real images
    for i in range(n_samples):
        if i < len(real_images):
            img = real_images[i].permute(1, 2, 0).cpu().numpy()
            axes[0, i].imshow(np.clip(img, 0, 1))
            axes[0, i].set_title(f'Real (Class {real_labels[i]})')
        axes[0, i].axis('off')
    
    # Plot synthetic images
    for i in range(n_samples):
        if i < len(synthetic_images):
            img = synthetic_images[i].permute(1, 2, 0).cpu().numpy()
            axes[1, i].imshow(np.clip(img, 0, 1))
            axes[1, i].set_title(f'Synthetic (Class {synthetic_labels[i]})')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Plot comparison
plot_image_comparison(train_images, synthetic_images, train_labels, synthetic_labels)

## 7. Evaluate Quality

Assess the quality of generated images using various metrics.

In [ ]:
# Evaluate quality
print("Evaluating synthetic image quality...")

# Use a subset of real images for comparison
comparison_images = val_images[:len(synthetic_images)]

quality_results = pipeline.evaluate_quality(
    synthetic_images=synthetic_images,
    real_images=comparison_images,
    return_detailed=True
)

# Display results
print("\nQuality Metrics:")
print("-" * 30)
for metric, value in quality_results['metrics'].items():
    print(f"{metric.upper()}: {value:.4f}")

if 'diversity' in quality_results:
    print("\nDiversity Metrics:")
    print("-" * 30)
    for metric, value in quality_results['diversity'].items():
        print(f"{metric}: {value:.4f}")

## 8. Save and Load Pipeline

Demonstrate how to save and load a trained pipeline for future use.

In [ ]:
# Save pipeline configuration and models
output_dir = Path("./pipeline_output")
output_dir.mkdir(exist_ok=True)

# Save pipeline
pipeline.save_pipeline(output_dir / "basic_pipeline")
print(f"Pipeline saved to {output_dir / 'basic_pipeline'}")

# Create configuration for reproducibility
config = PipelineConfig(
    config_name="basic_demo",
    encoder_config={
        'architecture': 'resnet18',
        'embedding_dim': 128,
        'pretrained': True
    },
    decoder_config={
        'decoder_type': 'autoencoder',
        'hidden_dims': [256, 512, 256]
    },
    smote_config={
        'k_neighbors': 5,
        'use_clustering': True
    },
    quality_config={
        'metrics': ['mse', 'ssim', 'psnr']
    }
)

config.save_config(output_dir / "config.json")
print(f"Configuration saved to {output_dir / 'config.json'}")

## 9. Class Distribution Analysis

Analyze how SMOTE helps balance the class distribution.

In [ ]:
# Analyze class distribution before and after SMOTE
original_dist = np.bincount(train_labels)
synthetic_dist = np.bincount(synthetic_labels)

# Combine original and synthetic
combined_labels = np.concatenate([train_labels, synthetic_labels])
combined_dist = np.bincount(combined_labels)

# Plot distribution comparison
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

classes = range(len(original_dist))

ax1.bar(classes, original_dist, alpha=0.7, color='blue')
ax1.set_title('Original Distribution')
ax1.set_xlabel('Class')
ax1.set_ylabel('Count')

ax2.bar(classes, synthetic_dist, alpha=0.7, color='red')
ax2.set_title('Synthetic Distribution')
ax2.set_xlabel('Class')
ax2.set_ylabel('Count')

ax3.bar(classes, combined_dist, alpha=0.7, color='green')
ax3.set_title('Combined Distribution')
ax3.set_xlabel('Class')
ax3.set_ylabel('Count')

plt.tight_layout()
plt.show()

print("Distribution Analysis:")
print(f"Original: {original_dist}")
print(f"Synthetic: {synthetic_dist}")
print(f"Combined: {combined_dist}")

## Conclusion

This notebook demonstrated the basic usage of the SMOTE for Images pipeline:

1. **Data Preparation**: Generated sample imbalanced dataset
2. **Pipeline Setup**: Configured encoder, decoder, SMOTE, and quality assessor
3. **Training**: Fitted the pipeline on training data
4. **Generation**: Created synthetic images for minority classes
5. **Evaluation**: Assessed quality using multiple metrics
6. **Analysis**: Compared class distributions before and after SMOTE

### Next Steps

- Try different decoder architectures (VAE, GAN, Diffusion)
- Experiment with different SMOTE parameters
- Use real datasets for more meaningful results
- Explore advanced quality metrics like FID and LPIPS

### Key Parameters to Tune

- **Embedding dimension**: Balance between information retention and efficiency
- **K-neighbors**: Controls diversity vs quality trade-off in SMOTE
- **Clustering**: Helps maintain semantic coherence
- **Decoder architecture**: Affects reconstruction quality